# Detecção de Landmarks

Um **landmark** (ou *keypoint*) é um ponto com significado fixo em um objeto: o canto do olho esquerdo, a ponta do nariz, o pulso. Diferente da detecção, o objeto tem sempre os mesmos $K$ pontos, e cada um tem identidade: não basta achar "um olho", é preciso dizer qual é o esquerdo. Com landmarks, um aplicativo alinha um rosto, aplica um filtro que acompanha a boca ou mede o ângulo do joelho em um agachamento.

Vamos treinar um detector de landmarks faciais como uma **regressão**: uma CNN que recebe a imagem e devolve as coordenadas de cada ponto. No fim, fazemos o ajuste fino de um YOLO de pose para encontrar os pontos de um animal, com poucas imagens.

In [ ]:
import os
import urllib.request
import zipfile

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, TensorDataset
from ultralytics import YOLO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## O dataset

Usamos o dataset *Facial Keypoints Detection*, do Kaggle: rostos em tons de cinza, com $96 \times 96$ pixels e até 15 pontos anotados. Cada linha do CSV tem o $x$ e o $y$ de cada ponto, em pixels, e a imagem como uma string com os 9216 pixels. Usamos só as linhas com os 15 pontos.

In [ ]:
IMAGE_SIZE = 96

csv_path = "data/facial-keypoints-dataset.csv"
if not os.path.exists(csv_path):
    gdown.download("https://drive.google.com/uc?id=14DTluCyfSr7MPkFaFVgbGn-Olm0zFAdK", "data/facial-keypoints-dataset.zip")
    with zipfile.ZipFile("data/facial-keypoints-dataset.zip") as z:
        z.extractall("data")

df = pd.read_csv(csv_path).dropna()
KEYPOINT_NAMES = [column[:-2] for column in df.columns[:-1:2]]   # tira o "_x" do nome
K = len(KEYPOINT_NAMES)

pixels = np.stack([np.array(image.split(), dtype=np.float32) for image in df["Image"]])
images = torch.tensor(pixels).reshape(-1, 1, IMAGE_SIZE, IMAGE_SIZE) / 255              # [N, 1, 96, 96]
points = torch.tensor(df.iloc[:, :-1].values, dtype=torch.float32).reshape(-1, K, 2)   # [N, 15, 2], (x, y) em pixels

print(images.shape, points.shape)
print(KEYPOINT_NAMES)

In [ ]:
def show_points(ax, image, points, color="lime"):
    ax.imshow(image[0], cmap="gray")
    ax.scatter(points[:, 0], points[:, 1], s=12, c=color)
    ax.axis("off")


fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, k in zip(axes, [0, 100, 200, 300]):
    show_points(ax, images[k], points[k])
    ax.set_title(f"imagem {k}")
plt.show()

Separamos 20% das imagens para teste.

In [ ]:
generator = torch.Generator().manual_seed(0)
order = torch.randperm(len(images), generator=generator)
n_test = len(images) // 5
test_idx, train_idx = order[:n_test], order[n_test:]

train_dataloader = DataLoader(TensorDataset(images[train_idx], points[train_idx]), batch_size=32, shuffle=True)
test_dataloader = DataLoader(TensorDataset(images[test_idx], points[test_idx]), batch_size=256)
print(f"{len(train_idx)} treino, {len(test_idx)} teste")

A medida é o **erro médio em pixels**, a distância entre o ponto previsto e o verdadeiro, na média sobre os pontos e as imagens. Como referência, o "modelo" mais simples ignora a imagem e responde sempre o **rosto médio** do treino. A rede precisa ser melhor que ele.

In [ ]:
def mean_error(predicted, true):
    return (predicted - true).norm(dim=-1).mean().item()   # distância euclidiana em pixels


mean_face = points[train_idx].mean(dim=0)   # [15, 2]
test_points = points[test_idx]
print(f"rosto médio: {mean_error(mean_face.expand_as(test_points), test_points):.2f} pixels")

## A rede

Uma CNN como as dos classificadores, terminando em uma camada linear com $2K = 30$ saídas, os $x$ e $y$ de cada ponto. As coordenadas são divididas por 96 para ficarem entre 0 e 1, e a perda é o erro quadrático médio.

In [ ]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
    )


class LandmarkNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(1, 32), nn.MaxPool2d(2),      # [N, 32, 48, 48]
            conv_block(32, 64), nn.MaxPool2d(2),     # [N, 64, 24, 24]
            conv_block(64, 128), nn.MaxPool2d(2),    # [N, 128, 12, 12]
            conv_block(128, 256), nn.MaxPool2d(2),   # [N, 256, 6, 6]
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256 * 6 * 6, 256),
            nn.ReLU(),
            nn.Linear(256, 2 * K),                   # [N, 30]
        )

    def forward(self, x):
        return self.head(self.features(x)).reshape(-1, K, 2)   # [N, 15, 2], entre 0 e 1


print(LandmarkNet()(images[:8]).shape)

## Treinamento

O laço é o dos classificadores, com o erro quadrático entre a saída e os pontos divididos por 96. A `predict` faz o caminho inverso, multiplicando a saída por 96 para voltar aos pixels.

In [ ]:
def train(model, train_dataloader, num_epochs, lr=1e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for images, points in train_dataloader:
            images, points = images.to(device), points.to(device)
            loss = F.mse_loss(model(images), points / IMAGE_SIZE)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        losses.append(total_loss / len(train_dataloader))
        if (epoch + 1) % 5 == 0:
            print(f"[{epoch + 1}/{num_epochs}] perda {losses[-1]:.5f}")

    return losses


def predict(model, images):
    model.eval()
    with torch.no_grad():
        return model(images.to(device)).cpu() * IMAGE_SIZE

In [ ]:
torch.manual_seed(42)
model = LandmarkNet().to(device)
losses = train(model, train_dataloader, num_epochs=30)

plt.plot(range(1, len(losses) + 1), losses)
plt.title("perda de treino")
plt.xlabel("época")
plt.ylabel("erro quadrático médio")
plt.show()

print(f"rede: {mean_error(predict(model, images[test_idx]), test_points):.2f} pixels")

Em verde, os pontos verdadeiros; em vermelho, os previstos, em imagens de teste.

In [ ]:
test_images = images[test_idx[:4]]
predicted = predict(model, test_images)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, image, true, pred in zip(axes, test_images, test_points, predicted):
    show_points(ax, image, true, "lime")
    ax.scatter(pred[:, 0], pred[:, 1], s=12, c="red")
    ax.set_title(f"erro {mean_error(pred, true):.2f} pixels")
plt.show()

A rede acerta a disposição geral do rosto, mas os pontos ficam alguns pixels fora do lugar, e o erro fica pouco abaixo do rosto médio. Com a taxa de aprendizado de $10^{-3}$ dos outros materiais, ela fica até pior que o rosto médio, por isso usamos $10^{-4}$.

A dificuldade está na camada linear: ela precisa aprender, para cada posição possível de cada ponto, como as 9216 características viram coordenadas, e com 1700 imagens ela aprende pouco além do rosto típico. Os três primeiros exercícios atacam justamente a falta de dados.

## Ajuste fino com o YOLO

Nossa rede recebe o rosto já recortado e centralizado. Em uma aplicação, alguém precisa encontrar o rosto antes, e são dois modelos rodando em sequência. O **YOLO de pose** faz as duas coisas de uma vez: além da caixa de cada objeto, ele prevê os pontos de cada um, e a imagem pode ter vários objetos.

Como no material de ajuste fino do YOLO, partimos de um modelo pré-treinado, aqui o `yolo26n-pose.pt`, treinado nos 17 pontos do corpo humano do COCO, e ajustamos para outro objeto. Usamos o **Tiger Pose**: 263 imagens de tigres com 12 pontos cada, poucas demais para treinar do zero.

In [ ]:
TIGER_DIR = os.path.abspath("data/tiger-pose")
if not os.path.exists(TIGER_DIR):
    zip_path, _ = urllib.request.urlretrieve("https://github.com/ultralytics/assets/releases/download/v0.0.0/tiger-pose.zip")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(TIGER_DIR)

TIGER_KEYPOINTS = [
    "nose", "head", "withers", "tail_base",
    "right_hind_hock", "right_hind_paw", "left_hind_paw", "left_hind_hock",
    "right_front_wrist", "right_front_paw", "left_front_wrist", "left_front_paw",
]

Cada imagem tem um arquivo de rótulo como os do detector, com uma linha por tigre: a classe, a caixa ($x$ e $y$ do centro, largura e altura) e depois o $x$ e o $y$ de cada um dos 12 pontos, tudo normalizado entre 0 e 1. Ligamos os pontos em um esqueleto para ver as anotações de alguns exemplos de treino.

In [ ]:
TIGER_SKELETON = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (3, 7), (7, 6), (2, 8), (8, 9), (2, 10), (10, 11)]


def draw_tiger(ax, image, box, points):
    width, height = image.size
    cx, cy, w, h = box * [width, height, width, height]
    points = points * [width, height]
    ax.imshow(image)
    ax.add_patch(plt.Rectangle((cx - w / 2, cy - h / 2), w, h, fill=False, color="yellow"))
    for a, b in TIGER_SKELETON:
        ax.plot(points[[a, b], 0], points[[a, b], 1], color="lime", linewidth=1.5)
    ax.scatter(points[:, 0], points[:, 1], s=15, c="red", zorder=3)
    ax.axis("off")


train_dir = os.path.join(TIGER_DIR, "images", "train")
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, name in zip(axes, sorted(os.listdir(train_dir))[::50]):
    label_path = os.path.join(TIGER_DIR, "labels", "train", name.rsplit(".", 1)[0] + ".txt")
    values = np.loadtxt(label_path, ndmin=2)[0]   # o primeiro tigre da imagem
    draw_tiger(ax, Image.open(os.path.join(train_dir, name)), values[1:5], values[5:].reshape(12, 2))
    ax.set_title(name)
plt.show()

O arquivo de configuração do dataset ganha duas linhas novas em relação ao do detector. O `kpt_shape` diz que são 12 pontos com 2 coordenadas cada, sem a visibilidade. O `flip_idx` diz qual ponto vira qual quando o YOLO espelha a imagem no aumento de dados: espelhada, a pata direita do tigre passa a ser a esquerda, e os índices precisam ser trocados. Montamos a lista a partir dos nomes.

In [ ]:
def mirror_name(name):
    if "left" in name:
        return name.replace("left", "right")
    return name.replace("right", "left")


flip_idx = [TIGER_KEYPOINTS.index(mirror_name(name)) for name in TIGER_KEYPOINTS]
print(flip_idx)

tiger_yaml = os.path.join(TIGER_DIR, "tiger.yaml")
with open(tiger_yaml, "w") as f:
    f.write(f"path: {TIGER_DIR}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("kpt_shape: [12, 2]\n")
    f.write(f"flip_idx: {flip_idx}\n")
    f.write("names:\n  0: tiger\n")
print(open(tiger_yaml).read())

O treino é o mesmo do detector. As métricas de pose usam a **OKS** (*object keypoint similarity*) no lugar da IoU: uma nota entre 0 e 1 para o quanto os pontos previstos estão perto dos verdadeiros, com uma tolerância proporcional ao tamanho do objeto. O mAP de pose é calculado como o das caixas, trocando a IoU pela OKS.

In [ ]:
runs_dir = os.path.abspath("runs")
tiger_model = YOLO("yolo26n-pose.pt")
tiger_model.train(data=tiger_yaml, epochs=50, imgsz=640, batch=16, project=runs_dir, name="tiger", exist_ok=True)

In [ ]:
tiger_model = YOLO(os.path.join(runs_dir, "tiger", "weights", "best.pt"))
metrics = tiger_model.val(data=tiger_yaml)
print(f"mAP50 das caixas: {metrics.box.map50:.3f}")
print(f"mAP50 da pose: {metrics.pose.map50:.3f}")
print(f"mAP50-95 da pose: {metrics.pose.map:.3f}")

In [ ]:
val_dir = os.path.join(TIGER_DIR, "images", "val")
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, name in zip(axes, sorted(os.listdir(val_dir))[::12]):
    image = Image.open(os.path.join(val_dir, name))
    result = tiger_model.predict(image, verbose=False)[0]
    box = result.boxes.xywhn[0].cpu().numpy()        # o tigre de maior confiança
    points = result.keypoints.xyn[0].cpu().numpy()   # [12, 2], entre 0 e 1
    draw_tiger(ax, image, box, points)
    ax.set_title(f"tigre, confiança {result.boxes.conf[0]:.2f}")
plt.show()

Com 210 imagens e cerca de cinco minutos de treino, o modelo encontra o tigre e os 12 pontos, e as previsões ficam parecidas com as anotações que vimos antes do treino. O mAP50 da pose chega ao máximo, mas é preciso desconfiar: as imagens de validação são quadros intercalados do mesmo vídeo do treino, com o mesmo tigre e o mesmo cenário, e a validação fica fácil demais. O mAP50-95, mais exigente, mostra que a localização ainda não é exata. Para saber como o modelo se sai com outros tigres, seria preciso um teste com vídeos diferentes.

## Exercícios

### Exercício 1: espelhamento

Treine espelhando cada imagem do lote com probabilidade 0,5, aplicando $x \to 95 - x$ aos pontos. Cuidado: os nomes seguem o ponto de vista da pessoa, e na imagem espelhada o olho esquerdo dela vira o direito. Troque os pontos `left_` e `right_` depois de espelhar. Compare o erro no teste com e sem a troca.

### Exercício 2: rótulos incompletos

Descartamos mais de dois terços do dataset, as imagens com só alguns pontos. Use todas as imagens e mude a perda para ignorar os pontos sem anotação, calculando o erro quadrático só onde o alvo existe. Mais imagens, mesmo incompletas, diminuem o erro?

### Exercício 3: transferência de aprendizado

Troque as camadas convolucionais por uma ResNet pré-treinada, como no material de transferência de aprendizado, repetindo o canal cinza três vezes na entrada. A rede pré-treinada erra menos?

### Exercício 4: o `flip_idx` errado

O arquivo de configuração que vem com o Tiger Pose usa `flip_idx: [0, 1, 2, ..., 11]`, sem trocar nenhum ponto. Treine o YOLO com esse `flip_idx` e compare o mAP de pose e as previsões das patas com as do modelo acima.